# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Croissant-format dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and examine metadata information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset from Croissant
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Citation: {getattr(dataset.metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review the available record sets (`@id`), their fields, and the fields' `@id` values as defined in the Croissant schema. This overview helps you select the relevant record set and field IDs for further analysis.

In [ ]:
# Print available record sets by `@id` and their fields/columns' `@id`s

info = dataset.metadata.to_jsonld()

# Helper function to recursively find all entities matching @type
def find_entities_of_type(info, type_name):
    found = []
    if isinstance(info, dict):
        if '@type' in info:
            if isinstance(info['@type'], list):
                if type_name in info['@type']:
                    found.append(info)
            else:
                if info['@type'] == type_name:
                    found.append(info)
        for v in info.values():
            found += find_entities_of_type(v, type_name)
    elif isinstance(info, list):
        for elem in info:
            found += find_entities_of_type(elem, type_name)
    return found

# List all record sets by @id
record_sets = find_entities_of_type(info, 'cr:RecordSet')
print("Record Sets found:")
for rs in record_sets:
    print(f"- @id: {rs.get('@id')}, name: {rs.get('name','<no name>')}")
    if 'field' in rs:
        print("  Fields (by @id):")
        for field in rs['field']:
            if isinstance(field, dict) and '@id' in field:
                print(f"    - {field['@id']}")
            elif isinstance(field, str):
                print(f"    - {field}")

if not record_sets:
    print("No record sets found in the schema metadata. Please check the Croissant file for correct record set definitions.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id` values displayed above. Here we extract all available record sets by their `@id`.

If the Croissant schema contains no record sets, this cell will not extract any data.

In [ ]:
# Extract all record sets into pandas DataFrames. Each record set is loaded by its `@id`.
dataframes = {}

if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"Loading records from record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Columns: {df.columns.tolist()}")
            print(df.head(2))
        else:
            print(f"  No records found for {rs_id}.")
else:
    print("No record sets found. Skipping data extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply basic processing steps such as filtering, normalizing, grouping, etc. Here we select a record set, a numeric field (`@id` as per overview), and demonstrate how to remove outliers, normalize, and group data, if possible.

**Note:** You must adjust the `analysis_record_set_id` and `numeric_field_id` to match values listed above if you wish to run this on your own copy and dataset.

In [ ]:
# Example only: set the record set and numeric field to analyze (replace with correct @ids from above cell)
analysis_record_set_id = None
numeric_field_id = None

# Try to auto-suggest a record set and numeric field for demonstration
import numpy as np

if dataframes:
    # Pick the first non-empty record set DataFrame with at least one numeric column
    for rsid, df in dataframes.items():
        numeric_candidates = df.select_dtypes(include=[np.number]).columns
        if len(numeric_candidates) > 0:
            analysis_record_set_id = rsid
            numeric_field_id = numeric_candidates[0]
            print(f"Auto-selected record set: {analysis_record_set_id}")
            print(f"Auto-selected numeric field: {numeric_field_id}")
            break

if analysis_record_set_id and numeric_field_id:
    df = dataframes[analysis_record_set_id]
    # Filter records for demonstration (e.g., those above the median)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt grouping by the first object-type column if available
    group_field_candidates = df.select_dtypes(include='object').columns
    if len(group_field_candidates) > 0:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable record set and numeric field found for EDA. Please inspect the data above and set `analysis_record_set_id` and `numeric_field_id` manually.")

## 5. Visualization
Visualize the distribution of a numeric field and the results of any groupings. This step may be adjusted by selecting different fields as required.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if analysis_record_set_id and numeric_field_id:
    df = dataframes[analysis_record_set_id]
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {analysis_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a grouping was generated, plot top N groups
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
        sns.barplot(data=top_groups, x=top_groups.columns[0], y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {top_groups.columns[0]}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped due to lack of suitable numeric field.")

## 6. Conclusion
We have demonstrated how to load and explore a Croissant-formatted mass spectrometry dataset using `mlcroissant`. Key steps included inspecting the schema, loading record sets and fields by `@id`, and performing basic exploratory data analysis and visualizations.

Remember: The Croissant format provides FAIR metadata and enables programmatic dataset discovery; always check and reference data entities by their `@id`. Further analyses can be performed based on your research questions and the structure of this and similar datasets.